# Crop Prediction System - 
fixed model training pipeline using MobileNetV2.

In [2]:
import os
import json
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [3]:
# 1. Collect Image Paths and Labels
DATA_DIR = "./Agricultural-crops/"

image_paths = []
labels = []

crop_folders = [f for f in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, f)) and not f.startswith(('.', 'myenv', 'venv'))]
crop_folders.sort()

for crop in crop_folders:
    crop_path = os.path.join(DATA_DIR, crop)
    for img_name in os.listdir(crop_path):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(DATA_DIR, crop, img_name))
            labels.append(crop)

df = pd.DataFrame({'Image_Path': image_paths, 'Crop_Label': labels})
print(f"Total Images Found: {len(df)}")
print(f"Total Crop Classes: {df['Crop_Label'].nunique()}")
df.head()

Total Images Found: 829
Total Crop Classes: 30


,Image_Path,Crop_Label
0,./Agricultural-crops/Cherry\image15.jpeg,Cherry
1,./Agricultural-crops/Cherry\image16.jpeg,Cherry
2,./Agricultural-crops/Cherry\image17.jpeg,Cherry
3,./Agricultural-crops/Cherry\image18.jpeg,Cherry
4,./Agricultural-crops/Cherry\image20.jpeg,Cherry


In [4]:
# 2. Encode Labels and Save Class Mapping
encoder = LabelEncoder()
y_int = encoder.fit_transform(labels)
class_names = list(encoder.classes_)

class_indices = {int(i): str(name) for i, name in enumerate(class_names)}
with open('class_indices.json', 'w') as f:
    json.dump(class_indices, f, indent=4)
print("Class indices saved to class_indices.json")

Class indices saved to class_indices.json


In [5]:
# 3. Load Images & Preprocess for MobileNetV2
X_list = []
for path in image_paths:
    img = Image.open(path).convert('RGB')
    img = img.resize((224, 224))
    img_arr = np.array(img, dtype=np.float32)
    img_arr = preprocess_input(img_arr)  # Scales to [-1, 1]
    X_list.append(img_arr)

X = np.array(X_list, dtype=np.float32)
print("X shape:", X.shape)

X shape: (829, 224, 224, 3)


In [6]:
# 4. Stratified Train-Test Split & One-Hot Encoding
X_train, X_test, y_train_int, y_test_int = train_test_split(
    X, y_int, test_size=0.2, random_state=42, stratify=y_int
)

num_classes = len(class_names)
y_train = to_categorical(y_train_int, num_classes=num_classes)
y_test = to_categorical(y_test_int, num_classes=num_classes)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

X_train: (663, 224, 224, 3) y_train: (663, 30)
X_test: (166, 224, 224, 3) y_test: (166, 30)


In [7]:
# 5. Data Augmentation & Model Building
train_datagen = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
train_datagen.fit(X_train)

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
output = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [8]:
# 6. Train Model (Phase 1 & Phase 2 Fine-Tuning)
callbacks = [
    ModelCheckpoint('crop_recommendation_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# Phase 1: Feature Extraction
history1 = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_test, y_test),
    epochs=20,
    callbacks=callbacks
)

# Phase 2: Fine-Tuning Top Layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_test, y_test),
    epochs=15,
    callbacks=callbacks
)

Epoch 1/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 853ms/step - accuracy: 0.2097 - loss: 3.2403
Epoch 1: val_accuracy improved from None to 0.49398, saving model to crop_recommendation_model.keras

Epoch 1: finished saving model to crop_recommendation_model.keras
21/21 ━━━━━━━━━━━━━━━━━━━━ 31s 1s/step - accuracy: 0.2097 - loss: 3.2403 - val_accuracy: 0.4940 - val_loss: 2.0953 - learning_rate: 0.0010
Epoch 2/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 752ms/step - accuracy: 0.5822 - loss: 1.5267
Epoch 2: val_accuracy improved from 0.49398 to 0.59639, saving model to crop_recommendation_model.keras

Epoch 2: finished saving model to crop_recommendation_model.keras
21/21 ━━━━━━━━━━━━━━━━━━━━ 20s 947ms/step - accuracy: 0.5822 - loss: 1.5267 - val_accuracy: 0.5964 - val_loss: 1.5544 - learning_rate: 0.0010
Epoch 3/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 749ms/step - accuracy: 0.6908 - loss: 1.0612
Epoch 3: val_accuracy improved from 0.59639 to 0.62048, saving model to crop_recommendation_model.keras

Epoch 3: finished

In [9]:
# 7. Evaluation & Save Check
best_model = tf.keras.models.load_model('crop_recommendation_model.keras')
test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
print(f"Final Test Loss: {test_loss:.4f}")
print(f"Final Test Accuracy: {test_acc*100:.2f}%")

y_pred = np.argmax(best_model.predict(X_test), axis=1)
print(classification_report(y_test_int, y_pred, target_names=class_names, zero_division=0))

Final Test Loss: 0.9166
Final Test Accuracy: 76.51%
6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 863ms/step
                     precision    recall  f1-score   support

             Cherry       0.44      0.67      0.53         6
       Coffee-plant       0.60      0.50      0.55         6
           Cucumber       0.75      1.00      0.86         6
   Fox_nut(Makhana)       1.00      0.80      0.89         5
              Lemon       1.00      1.00      1.00         6
         Olive-tree       0.67      1.00      0.80         6
Pearl_millet(bajra)       0.80      1.00      0.89         8
      Tobacco-plant       0.64      1.00      0.78         7
             almond       0.67      0.50      0.57         4
             banana       0.62      0.83      0.71         6
           cardamom       1.00      0.25      0.40         4
             chilli       1.00      0.60      0.75         5
              clove       1.00      0.33      0.50         6
            coconut       1.00      1.00      1.00    